In [ ]:
import sys
from pathlib import Path

project_root = str(Path.cwd().parent)
sys.path.append(project_root)

import matplotlib.pyplot as plt
import numpy as np
from core.linalg.decompositions import SVD
from core.linalg.matrix import Matrix
from PIL import Image

assets_dir = Path(project_root) / "assets"
image_path = assets_dir / "pku_boya.jpg"

# Load image in RGB
img = Image.open(image_path).convert("RGB")
img_resized = img.resize((512, 512))
img_np = np.array(img_resized, dtype=np.float64)

# Split into RGB channels and create a Matrix for each
channel_names = ["R", "G", "B"]
channel_data = {}
for i, name in enumerate(channel_names):
    channel_data[name] = Matrix(img_np[:, :, i])

print(f"Loaded RGB image. Shape: {img_np.shape}")
for name in channel_names:
    print(f"  {name} channel matrix shape: {channel_data[name].shape}")

# Visualize the loaded RGB image
plt.imshow(np.uint8(img_np))
plt.axis("off")
plt.title(f"Loaded: {image_path.name}")
plt.show()

In [ ]:
# Compute SVD for each RGB channel separately
channel_svd = {}
for name in channel_names:
    svd = SVD(channel_data[name], method="qr")
    channel_svd[name] = {
        "U": svd.U.data,
        "S": svd.S.data,
        "VT": svd.VT.data,
    }
    print(
        f"{name} channel SVD: U={channel_svd[name]['U'].shape}, "
        f"S={channel_svd[name]['S'].shape}, VT={channel_svd[name]['VT'].shape}"
    )


# Truncated SVD compression for RGB: compress each channel independently, then stack
def compress_image_rgb(channel_svd, original_img, channel_names, k):
    """Compress an RGB image by applying truncated SVD to each color channel."""
    compressed_channels = []
    for name in channel_names:
        U = channel_svd[name]["U"]
        S = channel_svd[name]["S"]
        VT = channel_svd[name]["VT"]

        m, n = U.shape[0], VT.shape[1]
        U_k = U[:, :k]
        Sigma_k = S[:k, :k]
        VT_k = VT[:k, :]
        A_k_data = U_k @ Sigma_k @ VT_k

        compressed_channels.append(A_k_data)

    # Stack channels back into an RGB image
    img_reconstructed = np.stack(compressed_channels, axis=-1)

    # Compute compression metrics
    original_size = m * n * 3
    compressed_size = k * (m + n + 1) * 3
    compression_ratio = compressed_size / original_size

    # Frobenius norm computed manually (np.linalg.norm with 'fro' doesn't support 3D arrays)
    diff = original_img - img_reconstructed
    error = np.sqrt(np.sum(diff**2)) / np.sqrt(np.sum(original_img**2))

    return img_reconstructed, compression_ratio, error

In [ ]:
k_values = [5, 10, 30, 80]

fig, axes = plt.subplots(1, len(k_values), figsize=(18, 5))

for ax, k in zip(axes, k_values):
    img_k, ratio, error = compress_image_rgb(channel_svd, img_np, channel_names, k)

    ax.imshow(np.uint8(np.clip(img_k, 0, 255)))
    ax.set_title(f"Rank $k={k}$\nStorage: {ratio:.1%}\nError: {error:.1%}", fontsize=12)
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
channel_colors = {"R": "red", "G": "green", "B": "blue"}
fig, ax1 = plt.subplots(figsize=(10, 5))

# Plot singular values for each channel
for name in channel_names:
    singular_values = np.diag(channel_svd[name]["S"])
    ax1.plot(
        singular_values,
        color=channel_colors[name],
        linewidth=1.5,
        alpha=0.8,
        label=f"{name} channel",
    )

ax1.set_xlabel("Singular Value Index ($i$)", fontsize=12)
ax1.set_ylabel(r"Singular Value $\sigma_i$", fontsize=12)
ax1.set_yscale("log")
ax1.legend(fontsize=10)

# Average cumulative energy across all channels
ax2 = ax1.twinx()
energy_ratios = []
for name in channel_names:
    singular_values = np.diag(channel_svd[name]["S"])
    energy_total = np.sum(singular_values**2)
    cumulative_energy = np.cumsum(singular_values**2) / energy_total
    energy_ratios.append(cumulative_energy)

avg_energy = np.mean(energy_ratios, axis=0)
ax2.plot(
    avg_energy, color="black", linestyle="--", linewidth=2, label="Avg Energy Ratio"
)
ax2.set_ylabel("Cumulative Energy Ratio (avg)", color="black", fontsize=12)
ax2.tick_params(axis="y", labelcolor="black")

plt.title("Singular Value Decay & Energy Cumulative Curve (RGB)", fontsize=14)
fig.tight_layout()
plt.grid(True, which="both", linestyle=":", alpha=0.5)
plt.show()